# Lab 2 — What the crew costs, and what you can change

**~25 minutes · nothing to fill in**

At the end of Lab 1, the hierarchical crew did the same work as the sequential crew and cost more. This
lab measures cost in more detail. You add a critic agent and measure it. Then you move the worker
agents to a different model and measure again. From the numbers, you decide whether either change was
worth it.

The point is not that critics are bad, or that one model is better. The point is that **you can
measure it**.

**Words used in this lab**

- **Critic:** an extra agent that checks another agent's work before anyone else sees it.
- **Worker:** an agent that does the main work. Here that is the researcher, the classifier and the
  writer from Lab 1.
- **Input and output price:** a model's price per token is different for what it reads (input, or
  prompt) and what it writes (output, or completion).

## 1 · Setup, and a way to compare runs

This lab uses the same gateway as Lab 1. The helper below keeps the numbers from each run side by side.
You compare them in a table, not from memory.

It also prices each run. It reads each model's prices from the gateway itself, so the `$` column is
what the run cost your key.

In [ ]:
import os, time, json, urllib.request
from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import tool

BASE = os.environ["OPENAI_BASE_URL"]
KEY  = os.environ["OPENAI_API_KEY"]

def gateway(path):
    req = urllib.request.Request(f"{BASE}{path}", headers={"Authorization": f"Bearer {KEY}"})
    return json.load(urllib.request.urlopen(req))

# $ per token, per model, as the gateway charges them. Read live, not typed in.
PRICES = {m["model_name"]: (m["model_info"].get("input_cost_per_token") or 0,
                            m["model_info"].get("output_cost_per_token") or 0)
          for m in gateway("/model/info")["data"]}

def model(name):
    return LLM(model=name, base_url=BASE, api_key=KEY, temperature=0)

RUNS = {}

def record(label, crew, seconds):
    # Count each LLM object ONCE. crew.usage_metrics adds up agent.llm's running
    # total once per agent, so three agents that share one LLM are counted three times.
    # A crew can also mix models, so each LLM's tokens are priced at that model's own rates.
    n = {"requests": 0, "prompt": 0, "completion": 0, "total": 0}
    cost = 0.0
    for llm in {id(a.llm): a.llm for a in crew.agents}.values():
        t = llm.get_token_usage_summary()
        n["requests"] += t.successful_requests
        n["prompt"] += t.prompt_tokens
        n["completion"] += t.completion_tokens
        n["total"] += t.total_tokens
        rate_in, rate_out = PRICES[llm.model]
        cost += t.prompt_tokens * rate_in + t.completion_tokens * rate_out
    RUNS[label] = {"s": round(seconds, 1), **n, "$": f"{cost:.5f}"}
    return RUNS[label]

def table():
    cols = ["s", "requests", "prompt", "completion", "total", "$"]
    print(f"{'run':<28}" + "".join(f"{c:>12}" for c in cols))
    for k, v in RUNS.items():
        print(f"{k:<28}" + "".join(f"{v[c]:>12}" for c in cols))

print("gateway:", BASE)

## 2 · Which models can you reach, and what do they cost?

Your key can reach more than one model. Read **both** price columns, input and output, not only the
total. Section 5 depends on the difference.

In [ ]:
print(f"{'model':<24}{'$/1M in':>10}{'$/1M out':>10}{'in + out':>10}")
for m in gateway("/models")["data"]:
    rate_in, rate_out = (r * 1_000_000 for r in PRICES[m["id"]])
    print(f"{m['id']:<24}{rate_in:>10.2f}{rate_out:>10.2f}{rate_in + rate_out:>10.2f}")

## 3 · Baseline: the desk, no critic

Three agents, in order, all on the standard lab model. Every other run is compared with this one.

In [ ]:
TICKETS = {
    "GB-T-4471": "My transfer of Rs 25,000 to my landlord failed twice, but my account was debited once.",
    "GB-T-4472": "I get 'invalid OTP' every time I log in on my new phone, since yesterday.",
    "GB-T-4473": "Please update my registered address. I have moved to Pune.",
    "GB-T-4474": "My savings account shows Rs 1,200 less than my passbook.",
}

# The four teams on the desk. They match the Global Bank services from Day 1.
TEAMS = ["Accounts", "Transactions", "Authentication", "Customer"]
TEAM_RULE = ("The category is the kind of problem, in a few words. "
             "The team must be one of: " + ", ".join(TEAMS) + ".")

@tool("ticket_lookup")
def ticket_lookup(ticket_id: str) -> str:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")

STANDARD = os.environ["OPENAI_MODEL"]

def build_desk(worker_model, critic=False, critic_model=None):
    w = model(worker_model)
    researcher = Agent(role="Ticket Researcher",
                       goal="Retrieve the ticket and state only the facts in it",
                       backstory="You pull the raw ticket and never guess beyond it.",
                       llm=w, tools=[ticket_lookup], allow_delegation=False)
    classifier = Agent(role="Support Triage Analyst",
                       goal="Classify a ticket and name the owning team",
                       backstory="You triage the Global Bank customer support desk.",
                       llm=w, allow_delegation=False)
    writer = Agent(role="Response Drafter",
                   goal="Write the first reply the bank customer will read",
                   backstory="You write to Global Bank customers plainly and never invent a timeline.",
                   llm=w, allow_delegation=False)

    t1 = Task(description="Retrieve ticket GB-T-4471 and list the facts it contains.",
              expected_output="A short bulleted list of facts.", agent=researcher)
    t2 = Task(description=f"Classify that ticket and name the owning team. {TEAM_RULE}",
              expected_output="'Category: <x>' and 'Team: <y>' on separate lines.", agent=classifier)
    t3 = Task(description="Draft a three-sentence first reply to the customer.",
              expected_output="Three sentences, no invented timeline.", agent=writer)

    agents, tasks = [researcher, classifier, writer], [t1, t2, t3]

    if critic:
        c = Agent(role="Quality Critic",
                  goal="Find anything in the draft reply that the ticket does not support",
                  backstory="You are the last check before a bank customer reads the reply. "
                            "You check every claim.",
                  llm=model(critic_model or worker_model), allow_delegation=False)
        t4 = Task(description="Review the draft reply. List anything it claims that the ticket does "
                              "not support. If it is clean, say so in one line.",
                  expected_output="A short list, or one line saying it is clean.", agent=c)
        agents.append(c); tasks.append(t4)

    return Crew(agents=agents, tasks=tasks, process=Process.sequential, verbose=False)

crew = build_desk(STANDARD)
s = time.time(); out = await crew.kickoff_async(); record("baseline", crew, time.time() - s)
print(out)
print()
table()

## 4 · Add the critic

One more agent and one more task. Run it, and look at what the extra check costs.

In [ ]:
crew_c = build_desk(STANDARD, critic=True)
s = time.time(); out_c = await crew_c.kickoff_async(); record("+ critic", crew_c, time.time() - s)
print(out_c)
print()
table()

**Decide before you read on.** Compare the two rows. The critic added requests, tokens and time. Did it
find anything wrong in the writer's reply?

If it said the reply was clean, you paid for a second check that this ticket did not need. So the real
question is not "is a critic worth it?". The real question is: **how often is the draft wrong, and what
does a wrong reply cost when a customer reads it?** You measure that rate over many tickets, not one.

## 5 · Move the workers to a different model

`gemma4-31b-lab` is from a different model family, Google's Gemma 4. Its headline price is close to
the standard model's: about $1 per million tokens, with the input and output prices added together. Put
the three workers on it, and keep the critic on the standard model.

**Predict before you run it:** will the `$` column go up, go down, or stay the same? Two things you
already have decide it:

- how section 2 splits each model's price between input and output, and
- how your `prompt` column compares with your `completion` column.

In [ ]:
crew_m = build_desk("gemma4-31b-lab", critic=True, critic_model=STANDARD)
s = time.time(); out_m = await crew_m.kickoff_async(); record("+ critic, gemma workers", crew_m, time.time() - s)
print(out_m)
print()
table()

## 6 · Read the three rows

You now have an argument you can take to your team:

- **Fewer tokens does not mean a smaller cost.** Compare the bottom row with the middle row. The Gemma
  run most likely used *fewer* tokens and still cost *more*. Both reasons are in section 2.
  - Models split text into tokens in different ways. The same prompts become a different number of
    tokens, so token counts do not compare across models.
  - A crew is mostly *input*. Every agent reads its role, its task and everything handed to it again,
    then writes a short answer. So the input price matters most, and Gemma's input price is far higher
    than the standard model's.
  - The similar headline price of about $1 hid both of these.
- **Price a model against your own work.** Only the `$` column compares across models. For a crew,
  "cheap" means a low *input* price, not a low total of the two prices.
- **Check quality yourself.** Compare the three replies. A worse answer is part of the cost. So is a
  better one.
- **The time changed too, and not in a steady way.** Run section 5 again. The seconds may change more
  than the tokens do.

### The failure modes to remember

- **Delegation in circles.** With `allow_delegation=True` on several agents, they can pass work to
  each other in circles. You pay for every pass. Turn it on for one agent at a time, and only on
  purpose.
- **A loose `expected_output`.** If a task does not say what it must produce, the next task guesses
  around whatever it gets. That is where a crew stops giving the same result each time.
- **A manager that does the work again.** In a hierarchical crew, the manager can re-read and re-answer
  its workers' output. You pay for both.
- **`crew.usage_metrics` over-counts a shared model.** It adds each agent's `llm` usage, and an `LLM`
  object's usage is a running total. Three agents that share one `LLM` are counted three times. That
  is why `record()` counts each `LLM` object once. Check this before you quote a crew's token count.

---

**You can now** say what a crew costs, where the cost comes from, and which change to try first. You
can back it up with numbers from your own run, not from a slide.

**Next:** Lab 3 builds the same desk in Google ADK.